# NB-205: Evaluation Metrics

Evaluate LLM performance with various metrics.

## Learning Objectives
- Perplexity
- BLEU score
- ROUGE score
- Human evaluation templates

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_metric
import numpy as np

print("Evaluation metrics ready")

## 1. Perplexity

In [ ]:
def compute_perplexity(model, tokenizer, texts):
    """Compute perplexity for texts."""

    total_loss = 0
    total_tokens = 0

    model.eval()

    with torch.no_grad():
        for text in texts:
            # TODO: Tokenize
            inputs = tokenizer(text, return_tensors="pt")

            # Forward pass
            outputs = model(**inputs, labels=inputs["input_ids"])

            # Accumulate loss
            total_loss += outputs.loss.item() * inputs["input_ids"].size(1)
            total_tokens += inputs["input_ids"].size(1)

    # Calculate perplexity
    avg_loss = total_loss / total_tokens
    perplexity = torch.exp(torch.tensor(avg_loss))

    return perplexity.item()

# Test
# model = AutoModelForCausalLM.from_pretrained("gpt2")
# tokenizer = AutoTokenizer.from_pretrained("gpt2")

# texts = ["Hello world", "This is a test"]
# ppl = compute_perplexity(model, tokenizer, texts)
# print(f"Perplexity: {ppl:.2f}")

print("Perplexity function defined")

## 2. BLEU Score

In [ ]:
def compute_bleu(predictions, references):
    """Compute BLEU score."""

    # TODO: Load BLEU metric
    try:
        bleu = load_metric("bleu")

        # Format for BLEU
        predictions = [pred.split() for pred in predictions]
        references = [[ref.split()] for ref in references]

        result = bleu.compute(predictions=predictions, references=references)

        return result
    except:
        print("BLEU metric not available, using sacrebleu")
        return None

# Test
predictions = ["the cat is on the mat", "hello world"]
references = ["the cat is on the mat", "hello there"]

result = compute_bleu(predictions, references)
if result:
    print(f"BLEU: {result['bleu']:.4f}")

## 3. ROUGE Score

In [ ]:
def compute_rouge(predictions, references):
    """Compute ROUGE score."""

    # TODO: Load ROUGE metric
    try:
        rouge = load_metric("rouge")

        result = rouge.compute(predictions=predictions, references=references)

        return result
    except:
        print("ROUGE metric not available")
        return None

# Test
predictions = ["The cat sat on the mat", "This is a summary"]
references = ["The cat is on the mat", "This is the summary"]

result = compute_rouge(predictions, references)
if result:
    print(f"ROUGE-L: {result['rougeL']:.4f}")

## 4. Human Evaluation Template

In [ ]:
def create_evaluation_prompt(prompt, response_a, response_b):
    """Create prompt for human evaluation."""

    eval_template = f"""Evaluate these two responses:

Prompt: {prompt}

Response A:
{response_a}

Response B:
{response_b}

Rate each on:
- Helpfulness (1-5)
- Accuracy (1-5)
- Quality (1-5)

Which is better? A or B?
"""

    return eval_template

# Test
prompt = "What is the capital of France?"
response_a = "The capital of France is Paris."
response_b = "Paris is France's capital."

eval_prompt = create_evaluation_prompt(prompt, response_a, response_b)
print(eval_prompt[:200] + "...")

## 5. Custom Metrics

In [ ]:
def compute_diversity(generated_texts):
    """Compute diversity of generated texts."""

    # TODO: Compute unique n-grams
    def get_ngrams(text, n):
        words = text.split()
        return set(tuple(words[i:i+n]) for i in range(len(words)-n+1))

    all_unigrams = set()
    all_bigrams = set()

    for text in generated_texts:
        all_unigrams.update(get_ngrams(text, 1))
        all_bigrams.update(get_ngrams(text, 2))

    total_unigrams = sum(len(get_ngrams(t, 1)) for t in generated_texts)
    total_bigrams = sum(len(get_ngrams(t, 2)) for t in generated_texts)

    return {
        "unigram_diversity": len(all_unigrams) / total_unigrams if total_unigrams > 0 else 0,
        "bigram_diversity": len(all_bigrams) / total_bigrams if total_bigrams > 0 else 0,
    }

# Test
texts = [
    "the cat is on the mat",
    "the dog is in the park",
    "birds fly high in the sky",
]

diversity = compute_diversity(texts)
print(f"Unigram diversity: {diversity['unigram_diversity']:.3f}")
print(f"Bigram diversity: {diversity['bigram_diversity']:.3f}")

## Exercise

1. Implement BERTScore evaluation
2. Add semantic similarity metrics
3. Create automated evaluation pipeline